# Recompute the safety tables from the public labels

This notebook downloads the judge's label files from the evaluation dataset `vincentnkq/aya-redteaming-6lang-eval` and recomputes the numbers of the write-up (`docs/article.md`): safe rate, refusal rate, over-refusal and chrF per language with Wilson intervals, and the paired change between two columns with a bootstrap interval and an exact sign test. It needs no GPU, no judge and no access to the model.

Requirements: Python 3.12 and `pip install huggingface_hub pandas scipy`. `scipy.stats.bootstrap` must accept `rng`, which needs scipy 1.15 or later. The tables in the write-up were built with scipy 1.18.1 and numpy 2.5.4; other versions may move a bootstrap bound in its last digit.

**Content warning:** the label files quote the harmful prompts. They do not hold the model's responses.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download
from scipy.stats import binomtest, bootstrap

REPO = 'vincentnkq/aya-redteaming-6lang-eval'
# a commit of the dataset to read; None reads the latest
REVISION = None
# 'labels' holds the runs under the served chat template, 'nopreamble/labels' the runs without its safety paragraph
TREE = 'labels'
# table column -> folder under TREE: the checkpoint's repo name and the first 12 characters of its commit
COLUMNS = {
    'baseline': 'tiny-aya-global--af89d219b53e',
    'drifted': 'tiny-aya-global-drift-merged--daccc14f6c19',
}
LANGUAGES = ['en', 'fr', 'es', 'ru', 'hi', 'ar']
LOW_RESOURCE = ['hi', 'ar']
BOOTSTRAP_SEED = 42
N_RESAMPLES = 1000

## Download

Only the label files are fetched. The column called `drifted` is the model after benign fine-tuning; the name is the one the pipeline's files use.

In [ ]:
root = Path(snapshot_download(REPO, repo_type='dataset', revision=REVISION, allow_patterns=[f'{TREE}/*']))
folders = sorted(path.name for path in (root / TREE).iterdir() if path.is_dir())
print('label folders in the dataset:', folders)

## Load

`<checkpoint>/<language>.jsonl` holds one row per harmful prompt, and `en.jsonl` also the XSTest rows. `<checkpoint>/benign/<language>.jsonl` holds the chrF of each held-out benign response. A label is `yes`, `no` or `unscored`.

In [ ]:
def read_rows(path):
    # split on newlines only: a prompt may hold other line separators
    return [json.loads(line) for line in path.read_text(encoding='utf-8').split('\n') if line.strip()]


def load(folder):
    here = root / TREE / folder
    paths = sorted(here.glob('*.jsonl')) + sorted(here.glob('benign/*.jsonl'))
    if not paths:
        raise FileNotFoundError(f'no label files under {TREE}/{folder}; folders found: {folders}')
    return [row for path in paths for row in read_rows(path)]


RAW = {column: load(folder) for column, folder in COLUMNS.items()}
labels = pd.concat(
    [pd.DataFrame(rows).assign(column=column) for column, rows in RAW.items()], ignore_index=True
)
labels['key'] = labels['language'] + '/' + labels['prompt_id']
judges = sorted({row['judge']['repo'] for rows in RAW.values() for row in rows if row['judge']})
print('judge:', judges)
print(labels.groupby(['column', 'set'], sort=False).size().unstack(fill_value=0).to_string())

## Rates

A response is safe when the judge's `response_harm` is `no`, and a refusal when `refusal` is `yes`. Over-refusal is the refusal rate on the XSTest prompts, which are safe to answer. An unscored pair is in no numerator and no denominator. Intervals are 95% Wilson intervals.

In [ ]:
def scored(frame, field, hit):
    # 1.0 or 0.0 where the judge answered yes or no, NaN where the pair is unscored
    return (frame[field] == hit).astype(float).where(frame[field].isin(['yes', 'no']))


labels['safe'] = scored(labels, 'response_harm', 'no')
labels['refusal'] = scored(labels, 'refusal', 'yes')
harmful = labels[labels['set'] == 'harmful']
xstest = labels[labels['set'] == 'xstest']
benign = labels[labels['set'] == 'benign']

low = [code for code in LANGUAGES if code in LOW_RESOURCE]
high = [code for code in LANGUAGES if code not in LOW_RESOURCE]
PLAIN = {code: [code] for code in LANGUAGES} | {'all': LANGUAGES}
GROUPS = PLAIN | {'+'.join(low): low, '+'.join(high): high}


def wilson(k, n):
    interval = binomtest(k, n).proportion_ci(confidence_level=0.95, method='wilson')
    return interval.low, interval.high


def rate_cell(values):
    values = values.dropna()
    k, n = int(values.sum()), len(values)
    if n == 0:
        return 'n = 0'
    interval = wilson(k, n)
    return f'{k / n:.1%} (n={n}; 95% CI {interval[0]:.1%} to {interval[1]:.1%})'


def chrf_cell(values):
    values = values.dropna()
    return f'{values.mean() * 100:.1f} (n={len(values)}; chrF1)' if len(values) else 'n = 0'


def table(frame, metric, cell, groups):
    cells = {
        column: {
            name: cell(frame[(frame['column'] == column) & frame['language'].isin(scope)][metric])
            for name, scope in groups.items()
        }
        for column in COLUMNS
    }
    return pd.DataFrame(cells).to_string()


unscored = harmful[harmful['safe'].isna()].groupby(['column', 'language'], sort=False).size()
print('unscored harmful pairs:', unscored.to_dict() or 'none')
print('\nSafe rate on harmful prompts\n' + table(harmful, 'safe', rate_cell, GROUPS))
print('\nRefusal rate on harmful prompts\n' + table(harmful, 'refusal', rate_cell, GROUPS))
print('\nOver-refusal on XSTest (English)\n' + table(xstest, 'refusal', rate_cell, {'en': ['en']}))
print('\nchrF on held-out benign prompts\n' + table(benign, 'chrf', chrf_cell, PLAIN))

## Change between two columns

The change is the later column minus the earlier one over the prompts scored in both. The interval is a percentile bootstrap over prompt pairs (1,000 resamples, seed 42), and p comes from an exact two-sided sign test on the pairs that differ. Pairs are ordered by language and prompt id before resampling, so the interval is the one in the published table.

In [ ]:
WORDS = {
    'safe': ('safer', 'less safe'),
    'over_refusal': ('more over-refusal', 'less over-refusal'),
    'chrf': ('higher chrF', 'lower chrF'),
}


def interval(samples, statistic, paired):
    result = bootstrap(
        samples,
        statistic,
        paired=paired,
        vectorized=True,
        n_resamples=N_RESAMPLES,
        confidence_level=0.95,
        method='percentile',
        rng=BOOTSTRAP_SEED,
    )
    return result.confidence_interval.low, result.confidence_interval.high


def paired_change(frame, metric, earlier, later, scope):
    # later minus earlier over the prompts scored in both columns, in key order
    part = frame[frame['language'].isin(scope)].dropna(subset=[metric])
    wide = part.pivot(index='key', columns='column', values=metric).reindex(columns=[earlier, later])
    wide = wide.dropna().sort_index()
    if len(wide) < 2:
        return None
    before, after = wide[earlier].to_numpy(dtype=float), wide[later].to_numpy(dtype=float)
    difference = after - before
    bounds = interval((after, before), lambda x, y, axis: np.mean(x - y, axis=axis), paired=True)
    # exact two-sided sign test on the non-zero differences
    up, down = int((difference > 0).sum()), int((difference < 0).sum())
    p = binomtest(up, up + down).pvalue if up + down else 1.0
    return len(wide), difference.mean(), bounds, p


def change_cell(change, words, n=None):
    if change is None:
        return 'n/a'
    pairs, delta, bounds, p = change
    # adding 0.0 turns a rounded -0.0 into 0.0
    points, lower, upper = (round(float(value) * 100, 1) + 0.0 for value in (delta, *bounds))
    word = 'no change' if points == 0 else words[points < 0]
    tested = '' if p is None else '; p < 0.001' if p < 0.001 else f'; p = {p:.3f}'
    size = n or f'{pairs} pairs'
    return f'{points:+.1f} points, {word} (95% CI {lower:+.1f} to {upper:+.1f}; n={size}{tested})'


earlier, later = list(COLUMNS)[:2]
print(f'Change from {earlier} to {later} (later minus earlier)')
for metric, frame, field, groups in (
    ('safe', harmful, 'safe', PLAIN),
    ('over_refusal', xstest, 'refusal', {'en': ['en']}),
    ('chrf', benign, 'chrf', PLAIN),
):
    for name, scope in groups.items():
        change = paired_change(frame, field, earlier, later, scope)
        print(f'{name:>4} {metric:<13}' + change_cell(change, WORDS[metric]))

## Hindi and Arabic against the other four languages

The difference in safe rate between the two groups within one column. The groups hold different prompts, so the bootstrap resamples each group on its own.

In [ ]:
def group_gap(frame, column):
    # mean of the low-resource group minus mean of the other languages; the two groups are not paired
    rows = frame[frame['column'] == column].dropna(subset=['safe'])
    first = rows[rows['language'].isin(low)]['safe'].to_numpy()
    second = rows[rows['language'].isin(high)]['safe'].to_numpy()
    if min(len(first), len(second)) < 2:
        return None, ''
    bounds = interval(
        (first, second), lambda x, y, axis: np.mean(x, axis=axis) - np.mean(y, axis=axis), paired=False
    )
    return (0, first.mean() - second.mean(), bounds, None), f'{len(first)} vs {len(second)}'


low_name, high_name = '+'.join(low), '+'.join(high)
print(f'{low_name} minus {high_name}, safe rate on harmful prompts')
for column in COLUMNS:
    gap, sizes = group_gap(harmful, column)
    print(f'{column:>10}  ' + change_cell(gap, (f'{low_name} safer', f'{low_name} less safe'), sizes))

## Optional: the same numbers through the package

With the code repository installed (`pip install git+https://github.com/vincentni/tiny-aya-safety-drift-map`), this cell reads the same rows through `driftmap.table`, the code that built the published tables, and checks that the pooled change agrees. Without the package it prints one line and does nothing.

In [ ]:
try:
    from driftmap.core.records import LabelRow
    from driftmap.table.rates import rate_row
    from driftmap.table.stats import paired_delta
except ImportError:
    print('driftmap is not installed; the cross-check is skipped')
else:
    typed = {
        column: [LabelRow.model_validate(row) for row in rows if row['set'] == 'harmful']
        for column, rows in RAW.items()
    }
    for column, rows in typed.items():
        pooled = rate_row('all', rows[0].checkpoint, 'safe', rows, rows[0].judge)
        print(f'{column:>10}  {pooled.k} of {pooled.n} safe, {pooled.rate:.1%}')
    theirs = paired_delta(typed[earlier], typed[later], 'safe')
    mine = paired_change(harmful, 'safe', earlier, later, LANGUAGES)
    assert theirs.n_pairs == mine[0] and np.allclose(
        [theirs.delta, *theirs.ci, theirs.p_value], [mine[1], *mine[2], mine[3]]
    )
    print('pooled change agrees with driftmap.table: ' + change_cell(mine, WORDS['safe']))